# Marketing Attribution ROI Dashboard

## Stage 3: Attribution Analysis

### Objective

This stage focuses on analyzing customer journeys across multiple marketing channels to understand how different touchpoints contribute to conversions and revenue.

We will build attribution models to measure channel effectiveness and support marketing ROI optimization.

---

### Key Questions

- Which marketing channels drive the most conversions?
- What is the customer journey before conversion?
- How do first-touch vs last-touch attribution compare?
- Which channels contribute most to revenue?
- Generate marketing KPIs for dashboard development.

In [79]:
import pandas as pd
import numpy as np

In [80]:
import os
print(os.getcwd())

c:\Users\HP\OneDrive\Documents\My Git Project\Marketing-Attribution-ROI-Dashboard\notebooks


In [81]:
import os
os.listdir("../data/processed")

['attribution_comparison.csv',
 'campaigns_cleaned.csv',
 'campaign_performance.csv',
 'channel_performance.csv',
 'customers_cleaned.csv',
 'events_cleaned.csv',
 'journey_data.csv',
 'products_cleaned.csv',
 'transactions_cleaned.csv']

In [82]:
import os
os.makedirs("../data/processed", exist_ok=True)

In [83]:
import os
print(os.listdir("../data/processed"))

['attribution_comparison.csv', 'campaigns_cleaned.csv', 'campaign_performance.csv', 'channel_performance.csv', 'customers_cleaned.csv', 'events_cleaned.csv', 'journey_data.csv', 'products_cleaned.csv', 'transactions_cleaned.csv']


In [84]:
# Load the cleaned CSV files into pandas DataFrames
campaigns = pd.read_csv("../data/processed/campaigns_cleaned.csv")
customers = pd.read_csv("../data/processed/customers_cleaned.csv")
events = pd.read_csv("../data/processed/events_cleaned.csv")
products = pd.read_csv("../data/processed/products_cleaned.csv")
transactions = pd.read_csv("../data/processed/transactions_cleaned.csv")

In [85]:
# Data check: Print the shape of each DataFrame to verify successful loading
print("Campaigns:", campaigns.shape)
print("Customers:", customers.shape)
print("Events:", events.shape)
print("Products:", products.shape)
print("Transactions:", transactions.shape)

Campaigns: (50, 7)
Customers: (100000, 7)
Events: (2000000, 12)
Products: (2000, 6)
Transactions: (103127, 9)


In [86]:
# Datetime preparation: Convert timestamp columns to datetime format for easier analysis
events["timestamp"] = pd.to_datetime(events["timestamp"])
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])
customers["signup_date"] = pd.to_datetime(customers["signup_date"])
campaigns["start_date"] = pd.to_datetime(campaigns["start_date"])
campaigns["end_date"] = pd.to_datetime(campaigns["end_date"])

In [87]:
events[["customer_id", "timestamp"]].head()

,customer_id,timestamp
0,43812,2021-01-14 13:35:43
1,71340,2021-12-03 21:36:50
2,59540,2021-12-27 08:25:15
3,3601,2022-01-22 15:06:54
4,92735,2021-05-10 12:03:09


In [88]:
# Sort events by customer_id and timestamp to prepare for attribution analysis
events = events.sort_values(["customer_id", "timestamp"])

In [89]:
events[["customer_id", "timestamp", "traffic_source"]].head(10)

,customer_id,timestamp,traffic_source
335888,1,2021-06-13 12:35:27,Organic
473478,1,2021-07-09 19:19:25,Paid Search
1784654,1,2021-10-18 18:22:42,Social
428621,1,2021-11-28 20:19:24,EMAIL
94193,1,2021-12-07 07:04:24,Paid Search
572035,1,2021-12-16 14:19:02,Paid Search
672060,1,2021-12-31 14:11:13,Email
1593995,1,2022-04-13 13:26:15,Email
965209,1,2022-08-01 07:13:40,Email
39685,1,2022-11-10 08:07:18,Social


## Conversion Definition

For this project, a conversion is defined as a completed transaction. Transaction records will therefore serve as the basis for attribution analysis.

In [90]:
# Define conversions from transactions table
transactions["is_conversion"] = 1

print("Total Transactions:", len(transactions))
print("Unique Converting Customers:", transactions["customer_id"].nunique())

Total Transactions: 103127
Unique Converting Customers: 64035


## Conversion Summary

The transactions dataset was used to define conversion events. Each transaction represents a successful conversion.

### Key Findings

- Total Transactions: 103,127
- Unique Converting Customers: 64,035

### Interpretation

The number of transactions exceeds the number of converting customers, indicating that some customers made multiple purchases. This suggests the presence of repeat customers and highlights the importance of analyzing customer lifetime value and repeat purchase behavior in addition to conversion volume.

This conversion definition serves as the foundation for subsequent attribution analysis, where marketing touchpoints will be linked to customer conversions.

In [91]:
# List of converting customers

converting_customers = transactions["customer_id"].unique()

print("Number of Converting Customers:", len(converting_customers))

Number of Converting Customers: 64035


## Identifying Converting Customers

To support attribution analysis, customers are classified as either converted or non-converted based on whether they completed at least one transaction.

This classification enables comparison of customer journeys and evaluation of channel effectiveness in driving conversions.

In [92]:
customers["converted"] = customers["customer_id"].isin(converting_customers)

customers["converted"].value_counts()

converted
True     64035
False    35965
Name: count, dtype: int64

## Customer Conversion Classification

Customers were classified as either converted or non-converted based on whether they completed at least one transaction.

### Results

- Converted Customers: 64,035
- Non-Converted Customers: 35,965

### Interpretation

Approximately 64% of customers completed at least one transaction, while 36% did not convert. This distinction provides a foundation for attribution analysis by enabling comparison between customers who successfully converted and those who did not.

The classification will support the evaluation of marketing channel effectiveness and customer journey patterns leading to conversion.

## Customer Journey Construction

Customer journeys represent the sequence of marketing touchpoints experienced by customers before conversion.

To prepare for attribution analysis, event records are organized chronologically for each customer. This enables identification of the first and last marketing touchpoints that influenced conversion decisions.

In [93]:
# Journey data preparation: Create a DataFrame that captures the customer journey with relevant columns
journey_data = events[
    ["customer_id", "timestamp", "traffic_source", "campaign_id"]
].copy()

journey_data.head()

,customer_id,timestamp,traffic_source,campaign_id
335888,1,2021-06-13 12:35:27,Organic,0
473478,1,2021-07-09 19:19:25,Paid Search,29
1784654,1,2021-10-18 18:22:42,Social,46
428621,1,2021-11-28 20:19:24,EMAIL,35
94193,1,2021-12-07 07:04:24,Paid Search,47


In [94]:
# Validate the journey data by checking the number of records and unique customers
print("Journey Records:", len(journey_data))
print("Unique Customers:", journey_data["customer_id"].nunique())

Journey Records: 2000000
Unique Customers: 100000


## Customer Journey Construction

To support attribution analysis, a structured customer journey dataset was created from the events data.

This dataset includes all customer interactions across marketing channels, ordered by time, to enable sequence-based analysis.

### Key Features Included

- `customer_id`: Unique identifier for each customer
- `timestamp`: Time of each interaction
- `traffic_source`: Marketing channel responsible for the touchpoint
- `campaign_id`: Associated marketing campaign

### Purpose

The goal of this step is to prepare a unified dataset that captures the full sequence of customer interactions before conversion. This forms the foundation for attribution modeling techniques such as first-touch and last-touch analysis.

In [95]:
journey_data = journey_data.sort_values(["customer_id", "timestamp"])

journey_data["touch_order"] = journey_data.groupby("customer_id").cumcount() + 1

journey_data.head(10)

,customer_id,timestamp,traffic_source,campaign_id,touch_order
335888,1,2021-06-13 12:35:27,Organic,0,1
473478,1,2021-07-09 19:19:25,Paid Search,29,2
1784654,1,2021-10-18 18:22:42,Social,46,3
428621,1,2021-11-28 20:19:24,EMAIL,35,4
94193,1,2021-12-07 07:04:24,Paid Search,47,5
572035,1,2021-12-16 14:19:02,Paid Search,28,6
672060,1,2021-12-31 14:11:13,Email,45,7
1593995,1,2022-04-13 13:26:15,Email,40,8
965209,1,2022-08-01 07:13:40,Email,18,9
39685,1,2022-11-10 08:07:18,Social,14,10


## Customer Journey Sequencing

Each customer interaction was assigned a sequential order based on timestamp using a grouped ranking approach.

This ensures that every touchpoint in the customer journey is correctly ordered from first interaction to last.

### Methodology

The sequencing was achieved using a group-wise cumulative count:

- Events were grouped by `customer_id`
- Each event was ordered chronologically by `timestamp`
- A `touch_order` column was generated to represent interaction sequence

### Why This Matters

This step is critical for attribution modeling because it enables:

- Identification of first-touch channels (touch_order = 1)
- Identification of last-touch channels (maximum touch_order per customer)
- Reconstruction of full customer journeys
- Preparation for multi-touch attribution models

### Output Structure

Each customer now has a clearly defined journey path showing how they interacted with different marketing channels over time.

In [96]:
# Fist touch attribution: Identify the first touchpoint for each customer
first_touch = journey_data.sort_values(["customer_id", "timestamp"]) \
                          .groupby("customer_id") \
                          .first() \
                          .reset_index()

first_touch.head()

,customer_id,timestamp,traffic_source,campaign_id,touch_order
0,1,2021-06-13 12:35:27,Organic,0,1
1,2,2021-01-30 06:32:34,Organic,0,1
2,3,2021-06-01 13:37:15,Organic,0,1
3,4,2021-01-08 09:19:08,Organic,0,1
4,5,2021-04-12 04:16:54,Organic,0,1


In [97]:
first_touch["traffic_source"].value_counts().head(10)

traffic_source
Organic        38936
Paid Search    19385
Email          14497
Social         14491
Direct          9645
ORGANIC         1167
PAID SEARCH      628
EMAIL            495
SOCIAL           455
DIRECT           301
Name: count, dtype: int64

In [98]:
# Clean traffic_source column: Remove leading/trailing spaces and standardize capitalization
journey_data["traffic_source"] = (
    journey_data["traffic_source"]
    .str.strip()
    .str.title()
)

In [99]:
# Validate the cleaning by checking unique traffic sources
first_touch = journey_data.sort_values(["customer_id", "timestamp"]) \
                          .groupby("customer_id", as_index=False) \
                          .first()

first_touch["traffic_source"].value_counts()

traffic_source
Organic        40103
Paid Search    20013
Email          14992
Social         14946
Direct          9946
Name: count, dtype: int64

## First-Touch Attribution Insights

The first-touch attribution analysis reveals how customers initially enter the marketing funnel.

### Key Findings

- Organic is the dominant acquisition channel, contributing the highest number of first-touch interactions.
- Paid Search ranks second, indicating strong intent-driven customer acquisition.
- Email and Social channels perform similarly, suggesting consistent mid-funnel awareness impact.
- Direct traffic is the lowest contributor, which is expected as it often represents returning or untracked users.

### Business Implication

Marketing investment should prioritize Organic and Paid Search channels for acquisition efficiency, while Email and Social can be optimized for engagement and nurturing strategies.

This analysis provides a foundational view of top-of-funnel performance across all marketing channels.

## Last-Touch Attribution

Last-touch attribution assigns 100% of the conversion credit to the final marketing interaction before the customer completes a transaction.

This model helps identify the channels that are most effective at driving final purchase decisions and closing conversions.

In [100]:
last_touch = journey_data.sort_values(["customer_id", "timestamp"]) \
                         .groupby("customer_id", as_index=False) \
                         .last()

last_touch.head()

,customer_id,timestamp,traffic_source,campaign_id,touch_order
0,1,2023-08-13 08:11:55,Organic,0,18
1,2,2023-12-30 14:51:13,Paid Search,27,19
2,3,2023-10-24 17:14:26,Organic,0,14
3,4,2023-12-12 05:16:14,Organic,0,23
4,5,2023-12-15 13:21:39,Social,22,11


In [101]:
last_touch["traffic_source"].value_counts()

traffic_source
Organic        40201
Paid Search    19779
Email          15068
Social         14912
Direct         10040
Name: count, dtype: int64

## Last-Touch Attribution Insights

The last-touch attribution model assigns full conversion credit to the final interaction before purchase.

### Key Findings

- Organic remains the strongest channel, indicating it not only attracts but also converts customers effectively.
- Paid Search shows strong performance as a high-intent conversion channel.
- Email performs better in last-touch than first-touch, suggesting strong nurturing and remarketing effectiveness.
- Social contributes consistently but does not dominate final conversion decisions.
- Direct traffic has the lowest contribution to last-touch conversions.

### Business Implications

Marketing efforts should focus on strengthening Organic and Paid Search channels for both acquisition and conversion. Email marketing demonstrates strong effectiveness in driving final purchase decisions and should be optimized for retention and remarketing strategies.

## Last-Touch Attribution Percentage Contribution

To better understand channel effectiveness, last-touch conversions were converted into percentage contributions.

### Results

- Organic: 40.20%
- Paid Search: 19.78%
- Email: 15.07%
- Social: 14.91%
- Direct: 10.04%

### Insight

Organic traffic dominates last-touch conversions, indicating strong brand authority and search visibility. Paid Search also performs strongly, confirming its role as a high-intent conversion channel. Email shows strong performance in closing conversions, highlighting its effectiveness in remarketing and customer engagement strategies.

In [102]:
# First touch attribution
first_counts = first_touch["traffic_source"].value_counts().rename("first_touch")
first_counts

traffic_source
Organic        40103
Paid Search    20013
Email          14992
Social         14946
Direct          9946
Name: first_touch, dtype: int64

In [103]:
# Last touch attribution
last_counts = last_touch["traffic_source"].value_counts().rename("last_touch")
last_counts

traffic_source
Organic        40201
Paid Search    19779
Email          15068
Social         14912
Direct         10040
Name: last_touch, dtype: int64

In [104]:
# Create a comparison DataFrame to analyze first touch vs last touch attribution counts
comparison = pd.concat([first_counts, last_counts], axis=1).fillna(0)

comparison["total"] = comparison["first_touch"] + comparison["last_touch"]

comparison

,first_touch,last_touch,total
traffic_source,,,
Organic,40103,40201,80304
Paid Search,20013,19779,39792
Email,14992,15068,30060
Social,14946,14912,29858
Direct,9946,10040,19986


In [105]:
# Percentage Contribution
comparison["first_touch_pct"] = (comparison["first_touch"] / comparison["first_touch"].sum()) * 100
comparison["last_touch_pct"] = (comparison["last_touch"] / comparison["last_touch"].sum()) * 100

comparison.sort_values("total", ascending=False)

,first_touch,last_touch,total,first_touch_pct,last_touch_pct
traffic_source,,,,,
Organic,40103,40201,80304,40.103,40.201
Paid Search,20013,19779,39792,20.013,19.779
Email,14992,15068,30060,14.992,15.068
Social,14946,14912,29858,14.946,14.912
Direct,9946,10040,19986,9.946,10.040


## First vs Last Touch Attribution Comparison

This analysis compares marketing channels based on their role in customer acquisition (first-touch) and conversion (last-touch).

### Key Insights

- Organic performs strongly in both acquisition and conversion stages, indicating full-funnel effectiveness.
- Paid Search shows balanced performance, contributing significantly to both user acquisition and conversions.
- Email is more dominant in last-touch than first-touch, highlighting its strength in nurturing and conversion.
- Social plays a supporting role across the funnel but is less dominant in conversion closure.
- Direct traffic primarily represents returning users and brand-driven conversions.

### Business Implication

Channels should not be evaluated solely on conversion counts. Instead, understanding their role across the customer journey enables more effective budget allocation and marketing strategy optimization.

## Linear Attribution Model

Unlike first-touch and last-touch attribution models, linear attribution distributes conversion credit equally across every touchpoint in the customer journey.

This approach recognizes that multiple marketing interactions may contribute to a conversion and assigns equal credit to each touchpoint involved.

### Objectives

- Assign equal credit across all customer touchpoints.
- Measure cumulative channel contribution.
- Compare results against first-touch and last-touch models.
- Provide a balanced view of marketing effectiveness.

In [106]:
# Calculate total touchpoints per customer
journey_data["total_touchpoints"] = journey_data.groupby(
    "customer_id"
)["customer_id"].transform("count")

journey_data[[
    "customer_id",
    "traffic_source",
    "touch_order",
    "total_touchpoints"
]].head()

,customer_id,traffic_source,touch_order,total_touchpoints
335888,1,Organic,1,18
473478,1,Paid Search,2,18
1784654,1,Social,3,18
428621,1,Email,4,18
94193,1,Paid Search,5,18


In [107]:
# Assign linear credit to each touchpoint based on the total number of touchpoints for that customer
journey_data["linear_credit"] = (
    1 / journey_data["total_touchpoints"]
)

journey_data[[
    "customer_id",
    "traffic_source",
    "touch_order",
    "total_touchpoints",
    "linear_credit"
]].head()

,customer_id,traffic_source,touch_order,total_touchpoints,linear_credit
335888,1,Organic,1,18,0.055556
473478,1,Paid Search,2,18,0.055556
1784654,1,Social,3,18,0.055556
428621,1,Email,4,18,0.055556
94193,1,Paid Search,5,18,0.055556


In [108]:
# Aggregate credit by chennel 
linear_attribution = (
    journey_data
    .groupby("traffic_source")["linear_credit"]
    .sum()
    .reset_index()
)

linear_attribution

,traffic_source,linear_credit
0,Direct,9987.776116
1,Email,14987.933977
2,Organic,40021.863145
3,Paid Search,19990.710576
4,Social,15011.716186


In [109]:
# Percentage Contribution of Linear Attribution
linear_attribution["percentage"] = (
    linear_attribution["linear_credit"]
    / linear_attribution["linear_credit"].sum()
) * 100

linear_attribution = linear_attribution.sort_values(
    by="percentage",
    ascending=False
)

linear_attribution

,traffic_source,linear_credit,percentage
2,Organic,40021.863145,40.021863
3,Paid Search,19990.710576,19.990711
4,Social,15011.716186,15.011716
1,Email,14987.933977,14.987934
0,Direct,9987.776116,9.987776


## Linear Attribution Analysis

The linear attribution model distributes conversion credit equally across all touchpoints in a customer's journey.

Unlike first-touch and last-touch models, this approach recognizes the contribution of every interaction that influenced a conversion.

### Methodology

- Count the total number of touchpoints for each customer.
- Assign equal credit to every touchpoint.
- Aggregate credit by marketing channel.
- Calculate percentage contribution of each channel.

This model provides a balanced assessment of marketing channel effectiveness across the entire customer journey.

## Linear Attribution Insights

The linear attribution model distributes conversion credit equally across all touchpoints within a customer's journey.

### Key Findings

- Organic contributes approximately 40% of total attribution credit, making it the most influential channel across the customer journey.
- Paid Search contributes approximately 20% of total credit, demonstrating strong and consistent influence.
- Email and Social channels each contribute around 15% of total credit.
- Direct traffic contributes approximately 10% of total credit.

### Business Implications

The similarity between first-touch, last-touch, and linear attribution results suggests that channel influence is relatively balanced throughout the customer journey. Organic and Paid Search remain the most impactful channels and should continue to receive strategic investment.

In [110]:
# Attribution Comparson Table
# First-touch percentages
first_pct = (
    first_touch["traffic_source"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("First Touch %")
)

# Last-touch percentages
last_pct = (
    last_touch["traffic_source"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("Last Touch %")
)

# Linear percentages
linear_pct = (
    linear_attribution
    .set_index("traffic_source")["percentage"]
    .rename("Linear %")
)

# Merge all models
attribution_comparison = pd.concat(
    [first_pct, last_pct, linear_pct],
    axis=1
)

attribution_comparison = attribution_comparison.round(2)

attribution_comparison

,First Touch %,Last Touch %,Linear %
traffic_source,,,
Organic,40.10,40.20,40.02
Paid Search,20.01,19.78,19.99
Email,14.99,15.07,14.99
Social,14.95,14.91,15.01
Direct,9.95,10.04,9.99


## Attribution Model Comparison

Three attribution models were evaluated to assess channel contribution across the customer journey:

- First-Touch Attribution
- Last-Touch Attribution
- Linear Attribution

### Results

| Channel | First Touch % | Last Touch % | Linear % |
|----------|-------------:|-------------:|---------:|
| Organic | 40.10 | 40.20 | 40.02 |
| Paid Search | 20.01 | 19.78 | 19.99 |
| Email | 14.99 | 15.07 | 14.99 |
| Social | 14.95 | 14.91 | 15.01 |
| Direct | 9.95 | 10.04 | 9.99 |

### Key Insights

- Organic is the most influential marketing channel, contributing approximately 40% across all attribution models.
- Paid Search contributes approximately 20% and remains the second strongest channel.
- Email and Social each contribute approximately 15%.
- Direct traffic contributes approximately 10%.

### Business Implications

The consistency across attribution methodologies suggests that channel influence remains relatively stable throughout the customer journey. This increases confidence in the reliability of the attribution analysis and supports continued investment in Organic and Paid Search channels.

## SQL Window Function Equivalent

The customer journey sequencing and attribution calculations performed in Python can also be implemented using SQL window functions.

This approach demonstrates how attribution analysis can be performed directly within a database environment.

In [111]:
# Import sqlite3
import sqlite3

In [112]:
# Create an in-memory SQLite database connection
conn = sqlite3.connect(":memory:")

In [113]:
# Load the journey_data DataFrame into the SQLite database as a table named "events"
journey_data.to_sql(
    "events",
    conn,
    index=False,
    if_exists="replace"
)

print("Events table loaded into SQLite")

Events table loaded into SQLite


In [114]:
# Customer touchpoints: Count the number of touchpoints for each customer using SQL
journey_data.groupby("customer_id")["customer_id"].transform("count")

335888     18
473478     18
1784654    18
428621     18
94193      18
           ..
1379048    28
946423     28
1829581    28
1357892    28
1790350    28
Name: customer_id, Length: 2000000, dtype: int64

In [115]:
# Total touchpoints per customer using SQL
query = """
SELECT
    customer_id,
    timestamp,
    traffic_source,

    COUNT(*) OVER (
        PARTITION BY customer_id
    ) AS total_touchpoints

FROM events
LIMIT 10;
"""

sql_touchpoints = pd.read_sql(query, conn)

sql_touchpoints

,customer_id,timestamp,traffic_source,total_touchpoints
0,1,2021-06-13 12:35:27,Organic,18
1,1,2021-07-09 19:19:25,Paid Search,18
2,1,2021-10-18 18:22:42,Social,18
3,1,2021-11-28 20:19:24,Email,18
4,1,2021-12-07 07:04:24,Paid Search,18
5,1,2021-12-16 14:19:02,Paid Search,18
6,1,2021-12-31 14:11:13,Email,18
7,1,2022-04-13 13:26:15,Email,18
8,1,2022-08-01 07:13:40,Email,18
9,1,2022-11-10 08:07:18,Social,18


In [116]:
# Linear Attribution Credit
query = """
SELECT
    customer_id,
    timestamp,
    traffic_source,

    ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY timestamp
    ) AS touch_order

FROM events
LIMIT 10;
"""

sql_touch_order = pd.read_sql(query, conn)

sql_touch_order

,customer_id,timestamp,traffic_source,touch_order
0,1,2021-06-13 12:35:27,Organic,1
1,1,2021-07-09 19:19:25,Paid Search,2
2,1,2021-10-18 18:22:42,Social,3
3,1,2021-11-28 20:19:24,Email,4
4,1,2021-12-07 07:04:24,Paid Search,5
5,1,2021-12-16 14:19:02,Paid Search,6
6,1,2021-12-31 14:11:13,Email,7
7,1,2022-04-13 13:26:15,Email,8
8,1,2022-08-01 07:13:40,Email,9
9,1,2022-11-10 08:07:18,Social,10


## SQL Window Function Result: Customer Journey Sequencing

The SQL output demonstrates the use of the `ROW_NUMBER()` window function to sequence customer interactions in chronological order.

### Output Explanation

| customer_id | timestamp | traffic_source | touch_order |
|-------------|-----------|----------------|-------------|
| 1 | 2021-06-13 12:35:27 | Organic | 1 |
| 1 | 2021-07-09 19:19:25 | Paid Search | 2 |
| 1 | 2021-10-18 18:22:42 | Social | 3 |
| 1 | 2021-11-28 20:19:24 | Email | 4 |
| 1 | 2021-12-07 07:04:24 | Paid Search | 5 |
| ... | ... | ... | ... |

### Key Insight

The `touch_order` column represents the sequential position of each marketing interaction within a customer's journey. This sequencing is generated using the SQL window function:

```sql
ROW_NUMBER() OVER (
    PARTITION BY customer_id
    ORDER BY timestamp
) AS touch_order

## Stage 3 Summary: Customer Journey & Attribution Analysis

In this stage, a complete multi-touch customer journey framework was developed to analyze how marketing channels contribute to conversions over time.

### 1. Customer Journey Construction

Customer interactions were sequenced chronologically using both Python and SQL window functions:

- Python: `groupby().cumcount() + 1`
- SQL: `ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY timestamp)`

This ensured each customer journey was properly ordered for attribution analysis.

---

### 2. Attribution Models Implemented

Three attribution models were developed and compared:

#### First-Touch Attribution
- Assigns full credit to the first interaction
- Focuses on acquisition channels

#### Last-Touch Attribution
- Assigns full credit to the final interaction
- Focuses on conversion-driving channels

#### Linear Attribution
- Distributes credit equally across all touchpoints
- Provides a balanced view of channel contribution

---

### 3. SQL Window Function Validation

SQL window functions were used to replicate Python logic and validate the analytical approach:

- `ROW_NUMBER()` → customer journey sequencing
- `COUNT() OVER (PARTITION BY customer_id)` → total touchpoints
- Derived linear attribution using SQL expressions

This demonstrates the ability to perform attribution analysis directly within a relational database environment.

---

### 4. Key Business Insights

- Organic is the strongest channel across all attribution models (~40% contribution)
- Paid Search consistently ranks second (~20% contribution)
- Email and Social channels contribute moderately (~15% each)
- Direct traffic contributes the least (~10%)

The consistency across all attribution models indicates stable and balanced channel performance throughout the customer journey.

---

### 5. Conclusion

Stage 3 successfully established a complete multi-touch attribution framework, enabling deeper understanding of marketing channel effectiveness across the full customer journey.

This forms the analytical foundation for advanced KPI development and dashboard visualization in the next stage of the project.

# Stage 4A: KPI Development & Dashboard Foundation

In [117]:
# Total customers
total_customers = customers["customer_id"].nunique()

# Converting customers
converting_customers = transactions["customer_id"].nunique()

# Conversion rate
conversion_rate = (converting_customers / total_customers) * 100

# Total touchpoints
total_touchpoints = len(events)

print("Total Customers:", total_customers)
print("Converting Customers:", converting_customers)
print("Conversion Rate:", round(conversion_rate, 2), "%")
print("Total Touchpoints:", total_touchpoints)

Total Customers: 100000
Converting Customers: 64035
Conversion Rate: 64.03 %
Total Touchpoints: 2000000


## KPI Interpretation

- The dataset contains 100,000 customers.
- 64,035 customers completed a conversion.
- This results in a 64.03% conversion rate, indicating a highly engaged customer base.
- The dataset contains 2 million marketing touchpoints, showing strong multi-touch behavior across the customer journey.

These KPIs form the foundation for all attribution and dashboard analysis in this project.

In [118]:
# Clean traffic_source column: Remove leading/trailing spaces and standardize capitalization
events["traffic_source"] = events["traffic_source"].str.upper()

In [119]:
mapping = {
    "ORGANIC": "Organic",
    "PAID SEARCH": "Paid Search",
    "SOCIAL": "Social",
    "EMAIL": "Email",
    "DIRECT": "Direct"
}

events["traffic_source"] = events["traffic_source"].replace(mapping)

In [120]:
# Traffic source distribution: Count the number of touchpoints for each traffic source
traffic_distribution = events["traffic_source"].value_counts()

print(traffic_distribution)

traffic_source
Organic        800489
Paid Search    399838
Social         300271
Email          299640
Direct         199762
Name: count, dtype: int64


In [121]:
# Calculate the percentage distribution of traffic sources
traffic_percentage = (traffic_distribution / traffic_distribution.sum()) * 100

print(round(traffic_percentage, 2))

traffic_source
Organic        40.02
Paid Search    19.99
Social         15.01
Email          14.98
Direct          9.99
Name: count, dtype: float64


In [122]:
# Channel Summary Table
channel_summary = pd.DataFrame({
    "touchpoints": traffic_distribution,
    "percentage": traffic_percentage
}).reset_index()

channel_summary.columns = ["traffic_source", "touchpoints", "percentage"]

channel_summary

,traffic_source,touchpoints,percentage
0,Organic,800489,40.02445
1,Paid Search,399838,19.99190
2,Social,300271,15.01355
3,Email,299640,14.98200
4,Direct,199762,9.98810


## Channel Engagement Insight

- Organic and Paid channels typically dominate customer interactions.
- Email and Social contribute significantly to mid-funnel engagement.
- Direct traffic represents lower but high-intent interactions.

This shows that marketing engagement is multi-channel and distributed across the funnel.

In [123]:
journey_data.columns

Index(['customer_id', 'timestamp', 'traffic_source', 'campaign_id',
       'touch_order', 'total_touchpoints', 'linear_credit'],
      dtype='str')

In [124]:
# % of linear attribution by traffic source
linear_pct = (
    journey_data.groupby("traffic_source")["linear_credit"].sum()
    / journey_data["linear_credit"].sum()
) * 100

linear_pct

traffic_source
Direct          9.987776
Email          14.987934
Organic        40.021863
Paid Search    19.990711
Social         15.011716
Name: linear_credit, dtype: float64

In [125]:
first_touch_df = journey_data[journey_data["touch_order"] == 1]

first_touch_pct = first_touch_df["traffic_source"].value_counts(normalize=True) * 100

first_touch_pct

traffic_source
Organic        40.103
Paid Search    20.013
Email          14.992
Social         14.946
Direct          9.946
Name: proportion, dtype: float64

In [126]:
last_touch_df = journey_data.sort_values("touch_order").groupby("customer_id").tail(1)

last_touch_pct = last_touch_df["traffic_source"].value_counts(normalize=True) * 100

last_touch_pct

traffic_source
Organic        40.201
Paid Search    19.779
Email          15.068
Social         14.912
Direct         10.040
Name: proportion, dtype: float64

In [127]:
comparison_table = pd.DataFrame({
    "First Touch %": first_touch_pct,
    "Last Touch %": last_touch_pct,
    "Linear %": linear_pct
}).fillna(0)

comparison_table.sort_values("Linear %", ascending=False)

,First Touch %,Last Touch %,Linear %
traffic_source,,,
Organic,40.103,40.201,40.021863
Paid Search,20.013,19.779,19.990711
Social,14.946,14.912,15.011716
Email,14.992,15.068,14.987934
Direct,9.946,10.040,9.987776


## Stage 4B: SQL Attribution & Performance Modeling

In this stage, we build a unified customer journey dataset using SQL. We apply JOIN operations and Window Functions to reconstruct user journeys and compute attribution weights.

We also analyze marketing channel performance and campaign effectiveness to support business decision-making and dashboard development.

In [128]:
# SQLite Database Setup
import sqlite3
conn = sqlite3.connect(":memory:")

In [129]:
# Load the DataFrames into the SQLite database
events.to_sql("events", conn, index=False, if_exists="replace")
customers.to_sql("customers", conn, index=False, if_exists="replace")
campaigns.to_sql("campaigns", conn, index=False, if_exists="replace")
transactions.to_sql("transactions", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")

2000

In [130]:
# Build customer journey dataset
query1 = """
SELECT
    e.customer_id,
    e.timestamp,
    e.traffic_source AS attribution_channel,

    COALESCE(cam.channel, 'No Campaign') AS campaign_channel,
    COALESCE(cam.objective, 'Not Assigned') AS objective,
    COALESCE(cam.target_segment, 'Not Assigned') AS target_segment,

    ROW_NUMBER() OVER (
        PARTITION BY e.customer_id
        ORDER BY e.timestamp
    ) AS touch_order,

    COUNT(*) OVER (
        PARTITION BY e.customer_id
    ) AS total_touchpoints,

    1.0 / COUNT(*) OVER (
        PARTITION BY e.customer_id
    ) AS linear_weight

FROM events e
LEFT JOIN campaigns cam
ON e.campaign_id = cam.campaign_id
"""

In [131]:
journey_data = pd.read_sql_query(query1, conn)

journey_data.head()

,customer_id,timestamp,attribution_channel,campaign_channel,objective,target_segment,touch_order,total_touchpoints,linear_weight
0,1,2021-06-13 12:35:27,Organic,No Campaign,Not Assigned,Not Assigned,1,18,0.055556
1,1,2021-07-09 19:19:25,Paid Search,Email,Acquisition,New Customers,2,18,0.055556
2,1,2021-10-18 18:22:42,Social,Social,Cross-sell,All,3,18,0.055556
3,1,2021-11-28 20:19:24,Email,Display,Acquisition,High Value,4,18,0.055556
4,1,2021-12-07 07:04:24,Paid Search,Affiliate,Cross-sell,High Value,5,18,0.055556


In [132]:
# Load the journey_data DataFrame into the SQLite database as a table named "journey_data"
journey_data.to_sql("journey_data", conn, index=False, if_exists="replace")

2000000

In [133]:
# Linear Attribution Credit by Channel
query2 = """
SELECT
    attribution_channel,
    SUM(linear_weight) AS linear_credit
FROM journey_data
GROUP BY attribution_channel
ORDER BY linear_credit DESC
"""

In [134]:
linear_summary = pd.read_sql_query(query2, conn)

linear_summary

,attribution_channel,linear_credit
0,Organic,40021.863145
1,Paid Search,19990.710576
2,Social,15011.716186
3,Email,14987.933977
4,Direct,9987.776116


In [135]:
linear_summary["percentage"] = (
    linear_summary["linear_credit"]
    / linear_summary["linear_credit"].sum()
) * 100

linear_summary

,attribution_channel,linear_credit,percentage
0,Organic,40021.863145,40.021863
1,Paid Search,19990.710576,19.990711
2,Social,15011.716186,15.011716
3,Email,14987.933977,14.987934
4,Direct,9987.776116,9.987776


In [136]:
# Channel Performance Summary: Count total touchpoints, unique customers, and sum of linear attribution credit by channel
query3 = """
SELECT
    attribution_channel,
    COUNT(*) AS total_touchpoints,
    COUNT(DISTINCT customer_id) AS unique_customers,
    ROUND(SUM(linear_weight), 2) AS attribution_credit
FROM journey_data
GROUP BY attribution_channel
ORDER BY attribution_credit DESC;
"""

channel_performance = pd.read_sql_query(query3, conn)

channel_performance

,attribution_channel,total_touchpoints,unique_customers,attribution_credit
0,Organic,800489,99966,40021.86
1,Paid Search,399838,98182,19990.71
2,Social,300271,95115,15011.72
3,Email,299640,95081,14987.93
4,Direct,199762,86262,9987.78


In [137]:
query4 = """
SELECT
    campaign_channel,
    objective,
    COUNT(*) AS total_touchpoints,
    COUNT(DISTINCT customer_id) AS unique_customers
FROM journey_data
GROUP BY
    campaign_channel,
    objective
ORDER BY total_touchpoints DESC;
"""

campaign_performance = pd.read_sql_query(query4, conn)

campaign_performance

,campaign_channel,objective,total_touchpoints,unique_customers
0,No Campaign,Not Assigned,1000251,99995
1,Paid Search,Cross-sell,119501,69731
2,Affiliate,Reactivation,79991,55120
3,Email,Reactivation,79917,55135
4,Email,Retention,60301,45191
5,Affiliate,Retention,60191,45282
6,Display,Reactivation,59913,45120
7,Social,Retention,59876,44995
8,Email,Acquisition,59716,45093
9,Display,Retention,40344,33224


In [138]:
# First touch attribution: Identify the first touchpoint for each customer
first_touch = (
    journey_data
    .sort_values("touch_order")
    .groupby("customer_id")
    .first()
)

first_touch_pct = (
    first_touch["attribution_channel"]
    .value_counts(normalize=True)
    * 100
)

In [139]:
# Last touch attribution: Identify the last touchpoint for each customer
last_touch = (
    journey_data
    .sort_values("touch_order")
    .groupby("customer_id")
    .last()
)

last_touch_pct = (
    last_touch["attribution_channel"]
    .value_counts(normalize=True)
    * 100
)

In [140]:
# Create a comparison table to analyze first touch, last touch, and linear attribution percentages by channel
comparison_table = pd.DataFrame({
    "First Touch %": first_touch_pct,
    "Last Touch %": last_touch_pct,
    "Linear %": linear_summary.set_index("attribution_channel")["percentage"]
}).fillna(0)

comparison_table

,First Touch %,Last Touch %,Linear %
attribution_channel,,,
Direct,9.946,10.040,9.987776
Email,14.992,15.068,14.987934
Organic,40.103,40.201,40.021863
Paid Search,20.013,19.779,19.990711
Social,14.946,14.912,15.011716


## Stage 4B Summary

Stage 4B focused on SQL-based marketing attribution modeling and performance analysis.

Using SQL JOINs and Window Functions, we reconstructed the full customer journey and computed linear attribution weights.

We also evaluated:

- Marketing channel performance
- Campaign effectiveness
- Attribution contribution across channels

These outputs provide a structured dataset for advanced visualization in Stage 5 (Power BI dashboard).

# KPI Development & Attribution Analysis Summary (Stage 4)

## Objective

Stage 4 focused on transforming cleaned customer journey data into meaningful business KPIs and building a complete marketing attribution framework using SQL-based data modeling.

The analysis applied SQL JOINs and Window Functions to reconstruct customer journeys and evaluate marketing performance using multiple attribution models:

- First-Touch Attribution  
- Last-Touch Attribution  
- Linear Attribution  

In addition, channel and campaign performance were analyzed to provide deeper marketing insights.

---

## Key Performance Indicators (KPIs)

- **Total Customers:** 100,000  
- **Converting Customers:** 64,035  
- **Conversion Rate:** 64.03%  
- **Total Touchpoints:** 2,000,000  

### KPI Insight

The dataset shows a high conversion rate of **64.03%**, indicating strong customer engagement and effective marketing reach.

The presence of **2 million touchpoints** confirms that customer journeys are multi-interaction and non-linear, making multi-touch attribution essential for accurate performance measurement.

---

## Channel Attribution Comparison

| Channel     | First Touch % | Last Touch % | Linear % |
|-------------|--------------:|-------------:|---------:|
| Organic     | 40.10         | 40.20        | 40.02    |
| Paid Search | 20.01         | 19.78        | 19.99    |
| Social      | 14.95         | 14.91        | 15.01    |
| Email       | 14.99         | 15.07        | 14.99    |
| Direct      | 9.95          | 10.04        | 9.99     |

---

## Channel Performance Analysis

The channel performance analysis evaluates each marketing channel based on:

- Total touchpoints  
- Unique customers  
- Attribution contribution  

### Key Findings:

- **Organic** is the strongest channel, contributing the highest engagement and attribution value.
- **Paid Search** consistently drives high-intent traffic and strong conversions.
- **Social and Email** play key roles in customer engagement and nurturing.
- **Direct traffic** reflects strong brand recall and high purchase intent.

This confirms that marketing performance is distributed across multiple channels, reinforcing the importance of multi-touch attribution.

---

## Campaign Performance Analysis

The campaign-level analysis evaluates performance based on campaign channels and objectives.

### Key Findings:

- Campaign channels such as **Email, Social, Display, and Affiliate** show varying levels of engagement across customer journeys.
- **Acquisition-focused campaigns** generate high touchpoints and early funnel activity.
- **Retention and cross-sell campaigns** contribute significantly to mid and late-stage conversions.
- Campaign objectives are strongly linked to customer lifecycle stages.

This provides visibility into how marketing campaigns contribute to different stages of the customer journey.

---

## Key Insights

### 1. Consistent Attribution Across Models
All three attribution models show similar distribution patterns, indicating a stable and well-balanced customer journey.

---

### 2. Organic is the Dominant Channel
Organic traffic contributes approximately **40% across all models**, making it the most influential channel for acquisition and conversion.

---

### 3. Paid Search is the Strongest Paid Channel
Paid Search contributes around **20%**, showing strong performance in capturing high-intent users.

---

### 4. Social and Email Support Engagement
Social and Email channels contribute between **14%–15%**, playing a key role in nurturing and mid-funnel engagement.

---

### 5. Direct Traffic Drives Conversions
Direct traffic contributes approximately **10%**, representing high-intent users who convert later in the journey.

---

## Business Conclusion

The analysis reveals a balanced multi-channel marketing ecosystem:

- **Organic** → Awareness and acquisition  
- **Paid Search** → High-intent conversion  
- **Social & Email** → Engagement and nurturing  
- **Direct** → Final conversion support  

Campaign analysis further shows that different campaign objectives (acquisition, retention, cross-sell) contribute uniquely across the funnel.

This confirms that **multi-touch attribution combined with channel and campaign analysis provides a more accurate and complete view of marketing performance than single-touch models**.

---

## Outcome of Stage 4

Stage 4 successfully:

- Built a SQL-based customer journey dataset  
- Implemented Window Functions for sequencing and attribution  
- Developed First-Touch, Last-Touch, and Linear attribution models  
- Performed channel performance analysis  
- Performed campaign performance analysis  
- Produced a dashboard-ready analytical dataset  

---

## Stage 4 Status

✔ KPI Development (Stage 4A)  
✔ SQL Attribution Modeling (Stage 4B)  
✔ Channel Performance Analysis  
✔ Campaign Performance Analysis  
✔ Power BI Readiness

In [141]:
journey_data.head()

,customer_id,timestamp,attribution_channel,campaign_channel,objective,target_segment,touch_order,total_touchpoints,linear_weight
0,1,2021-06-13 12:35:27,Organic,No Campaign,Not Assigned,Not Assigned,1,18,0.055556
1,1,2021-07-09 19:19:25,Paid Search,Email,Acquisition,New Customers,2,18,0.055556
2,1,2021-10-18 18:22:42,Social,Social,Cross-sell,All,3,18,0.055556
3,1,2021-11-28 20:19:24,Email,Display,Acquisition,High Value,4,18,0.055556
4,1,2021-12-07 07:04:24,Paid Search,Affiliate,Cross-sell,High Value,5,18,0.055556


In [ ]:
import os

processed_path = "../data/processed"
os.makedirs(processed_path, exist_ok=True)

In [ ]:
journey_data.to_csv(f"{processed_path}/journey_data.csv", index=False)

channel_performance.to_csv(
    f"{processed_path}/channel_performance.csv",
    index=False
)

campaign_performance.to_csv(
    f"{processed_path}/campaign_performance.csv",
    index=False
)

comparison_table.to_csv(
    f"{processed_path}/attribution_comparison.csv"
)

In [ ]:
from pathlib import Path

# Project root (one level above the notebooks folder)
project_root = Path.cwd().parent

# Data folders
raw_path = project_root / "data" / "raw"
processed_path = project_root / "data" / "processed"

# Create processed folder if it doesn't exist
processed_path.mkdir(parents=True, exist_ok=True)

print("Project Root:", project_root)
print("Processed Path:", processed_path)

Project Root: c:\Users\HP\OneDrive\Documents\My Git Project\Marketing-Attribution-ROI-Dashboard
Processed Path: c:\Users\HP\OneDrive\Documents\My Git Project\Marketing-Attribution-ROI-Dashboard\data\processed


# 📊 Stage 5: Business Intelligence Modeling & Dashboard Development

## Objective

Stage 5 focuses on transforming the cleaned and analytically enriched dataset into a **Business Intelligence-ready data model** using a **Star Schema architecture**. This stage bridges the gap between data analysis and interactive visualization by preparing structured fact and dimension tables optimized for Power BI.

The goal is to enable marketing stakeholders to explore customer journeys, attribution models, campaign performance, and ROI insights through an interactive dashboard.

---

## Business Context

Marketing teams require a unified view of customer interactions across multiple channels and campaigns. To support this, the dataset is modeled into a structured schema that allows:

- Efficient filtering and slicing of marketing data
- Fast performance in BI tools
- Scalable reporting across campaigns, channels, and customers

This stage ensures that the data is organized in a way that supports **decision-making at both executive and operational levels**.

---

## Star Schema Design

To meet Business Intelligence best practices, the dataset is structured into:

### Fact Table

**FactCustomerJourney**
- Contains customer touchpoints across the marketing funnel
- Stores attribution weights, timestamps, conversions, and revenue signals

### Dimension Tables

**DimCustomer**
- Customer demographics and acquisition data

**DimCampaign**
- Campaign metadata and targeting strategy

**DimProduct**
- Product catalog information

**DimDate**
- Time intelligence structure (created in Power BI)

---

## Key Activities in Stage 5

- Design and implementation of Star Schema data model  
- Creation of fact and dimension tables  
- Data transformation for BI compatibility  
- Preparation of datasets for Power BI integration  
- Structuring data for interactive reporting and dashboards  

---

## Expected Outcome

By the end of Stage 5, the project will deliver:

- A fully structured **Star Schema data model**
- Power BI-ready datasets stored in `data/processed`
- A foundation for building interactive dashboards
- A system optimized for analyzing:
  - Conversion funnels  
  - ROI performance  
  - Campaign effectiveness  
  - Channel attribution comparisons  

---

## Next Step

Once the data model is finalized, the next phase will focus on:

### 📊 Power BI Dashboard Development
- Conversion Funnel Visualization  
- ROI Scatter Plot  
- Channel Comparison Bar Chart  
- Attribution Model Toggle (First-Touch, Last-Touch, Linear)  
- Executive and Performance Marketer dashboards  

---

## Summary

Stage 5 represents the transition from **data analysis to business intelligence reporting**, ensuring that all previous analytical work is structured into a scalable and visualization-ready model for decision-making.

In [ ]:
# Create a comprehensive fact table for customer journeys, including campaign and transaction details
query_fact_clean = """
SELECT
    e.customer_id,
    e.campaign_id,
    e.timestamp,

    e.traffic_source AS attribution_channel,

    COALESCE(c.channel, 'No Campaign') AS campaign_channel,
    COALESCE(c.objective, 'Not Assigned') AS objective,
    COALESCE(c.target_segment, 'Not Assigned') AS target_segment,

    COALESCE(t.gross_revenue, 0) AS gross_revenue,
    COALESCE(t.is_conversion, 0) AS is_conversion,

    ROW_NUMBER() OVER (
        PARTITION BY e.customer_id
        ORDER BY e.timestamp
    ) AS touch_order,

    COUNT(*) OVER (
        PARTITION BY e.customer_id
    ) AS total_touchpoints,

    ROUND(
        1.0 / COUNT(*) OVER (PARTITION BY e.customer_id),
        6
    ) AS linear_weight

FROM events e

LEFT JOIN campaigns c
    ON e.campaign_id = c.campaign_id

LEFT JOIN transactions t
    ON e.customer_id = t.customer_id
    AND t.is_conversion = 1;
"""

FactCustomerJourney = pd.read_sql_query(query_fact_clean, conn)

FactCustomerJourney.head()

,customer_id,campaign_id,timestamp,attribution_channel,campaign_channel,objective,target_segment,gross_revenue,is_conversion,touch_order,total_touchpoints,linear_weight
0,1,0,2021-06-13 12:35:27,Organic,No Campaign,Not Assigned,Not Assigned,0.0,0,1,18,0.055556
1,1,29,2021-07-09 19:19:25,Paid Search,Email,Acquisition,New Customers,0.0,0,2,18,0.055556
2,1,46,2021-10-18 18:22:42,Social,Social,Cross-sell,All,0.0,0,3,18,0.055556
3,1,35,2021-11-28 20:19:24,Email,Display,Acquisition,High Value,0.0,0,4,18,0.055556
4,1,47,2021-12-07 07:04:24,Paid Search,Affiliate,Cross-sell,High Value,0.0,0,5,18,0.055556


In [ ]:
# Dimension Customer Table
DimCustomer = customers.copy()

DimCustomer.head()

,customer_id,signup_date,country,age,gender,loyalty_tier,acquisition_channel,converted
0,1,2021-04-08,BR,48,Male,Bronze,Referral,False
1,2,2023-04-28,IN,36,Female,Silver,Organic,True
2,3,2022-12-18,UK,35,Female,Silver,Organic,False
3,4,2022-04-26,US,45,Male,Silver,Paid Search,False
4,5,2022-04-20,IN,53,Male,Silver,Organic,False


In [ ]:
# Dimension Campaign Table
DimCampaign = campaigns.copy()

DimCampaign.head()

,campaign_id,channel,objective,start_date,end_date,target_segment,expected_uplift
0,1,Paid Search,Cross-sell,2021-10-25,2021-11-26,Deal Seekers,0.022
1,2,Email,Retention,2021-10-24,2021-12-24,Deal Seekers,0.116
2,3,Email,Reactivation,2023-10-08,2023-11-30,Churn Risk,0.100
3,4,Display,Reactivation,2022-07-25,2022-10-07,Deal Seekers,0.111
4,5,Social,Acquisition,2022-07-09,2022-09-29,New Customers,0.144


In [ ]:
# Dimension Product Table
DimProduct = products.copy()

DimProduct.head()

,product_id,category,brand,base_price,launch_date,is_premium
0,1,Grocery,Brand_58,14.19,2021-08-02,0
1,2,Fashion,Brand_1,25.80,2021-09-14,0
2,3,Electronics,Brand_70,165.46,2021-01-18,1
3,4,Fashion,Brand_56,75.45,2023-03-03,1
4,5,Sports,Brand_1,72.50,2022-04-19,1



---

## Fact Table

### FactCustomerJourney

This is the central fact table containing all customer touchpoints across the marketing funnel.

**Grain:** One row per customer interaction (touchpoint)

### Key Columns:

- customer_id  
- campaign_id  
- timestamp  
- attribution_channel  
- campaign_channel  
- objective  
- target_segment  
- touch_order  
- total_touchpoints  
- linear_weight  
- gross_revenue  
- is_conversion  

### Purpose:

- Tracks full customer journey across channels  
- Enables multi-touch attribution analysis  
- Supports funnel and conversion tracking  
- Powers ROI and campaign effectiveness analysis  

---

## Dimension Tables

### DimCustomer

Contains customer demographic and acquisition details.

- customer_id (Primary Key)  
- country  
- age  
- gender  
- loyalty_tier  
- acquisition_channel  

---

### DimCampaign

Contains marketing campaign metadata.

- campaign_id (Primary Key)  
- channel  
- objective  
- target_segment  
- expected_uplift  

---

### DimProduct

Contains product catalog information.

- product_id (Primary Key)  
- category  
- brand  
- base_price  
- is_premium  
- launch_date  

---

### DimDate

Time intelligence table (created in Power BI).

- Date  
- Year  
- Month  
- Quarter  
- Week  
- Day  

---

## Relationships

| Table | Relationship |
|------|-------------|
| FactCustomerJourney → DimCustomer | customer_id |
| FactCustomerJourney → DimCampaign | campaign_id |
| FactCustomerJourney → DimProduct | product_id |
| FactCustomerJourney → DimDate | timestamp |

---

## Purpose of Star Schema

This model is designed to:

- Improve Power BI performance  
- Enable easy slicing and filtering  
- Support attribution modeling (First-Touch, Last-Touch, Linear)  
- Enable funnel analysis and ROI tracking  
- Provide a scalable structure for marketing analytics  

---

## Outcome

The Star Schema serves as the foundation for the Power BI dashboard, enabling:

- Conversion funnel visualization  
- Campaign performance analysis  
- Channel attribution comparison  
- Executive-level marketing insights  